In [13]:
#-----------------------------
# 02_train_vgg16.ipynb
#-----------------------------

# Importations
import numpy as np
import torch
import os
import torch.nn as nn
from torchvision.models import vgg16, VGG16_Weights
from mlflow.tracking import MlflowClient
from mlflow.entities import ViewType
from torch.utils.data import TensorDataset, DataLoader, random_split
from torchvision import models, transforms
import torch.optim as optim
import mlflow
import mlflow.pytorch


In [5]:
# -----------------------------
# Paramètres
# -----------------------------
#le modèle voit 32 images puis calcule la perte et met à jour les poids.
#32 est un compromis classique pour les GPU moyens.
BATCH_SIZE = 16
#Nombre de passages complets sur tout le dataset(si trop d'epochs le modèle peut apprendre par cœur les images.)
EPOCHS = 15
#Détermine la taille du pas pour la mise à jour des poids à chaque backpropagation.
#(On ne veut pas changer brutalement les poids → learning rate petit.)
LEARNING_RATE = 1e-4
#Permet de passer les calculs sur GPU si disponible, sinon CPU (Accélération calcul)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMAGE_SIZE = 224  # Doit correspondre au preprocessing
NUM_CLASSES = 2

# Résolution robuste du dossier processed (remonte l'arborescence depuis le cwd)
def find_processed_dir():
    cur = os.getcwd()
    while True:
        candidate = os.path.join(cur, "data", "processed")
        if os.path.isdir(candidate):
            return os.path.abspath(candidate)
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    # fallback: try relative to this notebook file location if possible
    # (Jupyter notebooks may start with different working dirs)
    possible = os.path.abspath(os.path.join('..', '..', 'data', 'processed'))
    if os.path.isdir(possible):
        return possible
    raise FileNotFoundError(f'Could not find data/processed starting from cwd={os.getcwd()}')

PROCESSED_DIR = find_processed_dir()



In [6]:
# -----------------------------
# Chargement des données
# -----------------------------
# Si grayscale → 1 canal
# Transforme en tenseur PyTorch et ajoute dimension canal
# Chemin vers le dossier processed


# Charger les fichiers numpy
images = np.load(os.path.join(PROCESSED_DIR, "images.npy"))  # shape = [N,224,224,1]
labels = np.load(os.path.join(PROCESSED_DIR, "labels.npy"))
X = torch.tensor(images, dtype=torch.float32).unsqueeze(1)  # shape = [N,1,224,224]
y = torch.tensor(labels, dtype=torch.long)

# Créer Dataset et DataLoader
dataset = TensorDataset(X, y)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Train batch shape:", next(iter(train_loader))[0].shape)
print("Validation batch shape:", next(iter(val_loader))[0].shape)



Train batch shape: torch.Size([16, 1, 224, 224, 1])
Validation batch shape: torch.Size([16, 1, 224, 224, 1])


In [ ]:
#-----------------------------
# Définition du modèle VGG16
#-----------------------------
#contient toutes les couches de VGG16 avec des poids déjà appris.
model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)


#première couche convolutionnelle du VGG16.
old_conv = model.features[0]
# Adapter la première couche pour 1 canal
# permet de capturer des motifs locaux
#Pour ne pas réduire la résolution trop vite.
#Pour que la taille de l’image reste la même après la convolution.(1)
new_conv = nn.Conv2d(
    in_channels=1,
    out_channels=old_conv.out_channels,
    kernel_size=old_conv.kernel_size,
    stride=old_conv.stride,
    padding=old_conv.padding
)
with torch.no_grad():
    new_conv.weight[:, 0, :, :] = old_conv.weight.mean(dim=1)
model.features[0] = new_conv

# Modifier la dernière couche pour 2 classes
model.classifier[6] = nn.Linear(model.classifier[6].in_features, 2)
model = model.to(DEVICE)


In [8]:
# Loss et optimizer
#une mesure de la différence entre les probabilités prédites par le modèle et les vraies étiquettes
criterion = nn.CrossEntropyLoss()
#Un optimiseur sert à mettre à jour les poids du réseau pour minimiser la loss
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [9]:
# -----------------------------
# Configuration MLflow
# -----------------------------
mlflow.set_tracking_uri("file:./mlruns")  # dossier mlruns dans le projet
mlflow.set_experiment("VGG16_model")


2026/02/02 22:24:08 INFO mlflow.tracking.fluent: Experiment with name 'VGG16_model' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///c:/Users/USER/Desktop/ecg-classification/src/train/mlruns/723800116184727856', creation_time=1770067448620, experiment_id='723800116184727856', last_update_time=1770067448620, lifecycle_stage='active', name='VGG16_model', tags={}>

In [10]:
# Vérifie si un run fantôme existe
if mlflow.active_run() is not None:
    print("Un run fantôme détecté :", mlflow.active_run().info.run_id)
    
    # Solution compatible MLflow 2.x / 3.x
    mlflow.active_run()._run_id = None  # réinitialise le run actif
    print("Run fantôme réinitialisé ✅")


In [11]:
# -----------------------------
# Entraînement avec MLflow
# -----------------------------
with mlflow.start_run(run_name="VGG16_Run") as run:

    mlflow.log_param("model", "VGG16")
    mlflow.log_param("epochs", EPOCHS)
    mlflow.log_param("batch_size", BATCH_SIZE)
    mlflow.log_param("learning_rate", LEARNING_RATE)
    mlflow.log_param("optimizer", "Adam")

    for epoch in range(EPOCHS):
       # met le modèle en mode entraînement (activation de dropout et batchnorm).
        model.train()
         #Initialisation des variables pour calculer loss et accuracy sur l’epoch.
        train_loss, correct, total = 0, 0, 0
        #On récupère les batches depuis le train_loader.
        for inputs, targets in train_loader:
            #envoie les données sur GPU ou CPU.
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            #Cas particulier : certaines images ont une dimension supplémentaire.squeeze(-1) supprime cette dimension inutile.
            if inputs.dim() == 5 and inputs.size(-1) == 1:
                    inputs = inputs.squeeze(-1)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total += targets.size(0)
            correct += (predicted == targets).sum().item()

        train_loss /= train_size
        train_acc = correct / total

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
                if inputs.dim() == 5 and inputs.size(-1) == 1:
                    inputs = inputs.squeeze(-1)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs, 1)
                val_total += targets.size(0)
                val_correct += (predicted == targets).sum().item()

        val_loss /= val_size
        val_acc = val_correct / val_total

        print(f"Epoch [{epoch+1}/{EPOCHS}] Train Loss: {train_loss:.4f} Train Acc: {train_acc:.4f} "
              f"Val Loss: {val_loss:.4f} Val Acc: {val_acc:.4f}")

        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("train_acc", train_acc, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
        mlflow.log_metric("val_acc", val_acc, step=epoch)
 # ----- Sauvegarde du modèle -----
    model.eval()
    mlflow.pytorch.log_model(
        pytorch_model=model,
        artifact_path="VGG16_model",
    )

    print("Run MLflow terminé avec succès :", run.info.run_id)

Epoch [1/15] Train Loss: 0.3707 Train Acc: 0.8354 Val Loss: 0.1995 Val Acc: 0.8942
Epoch [2/15] Train Loss: 0.1519 Train Acc: 0.9443 Val Loss: 0.1690 Val Acc: 0.9231
Epoch [3/15] Train Loss: 0.1294 Train Acc: 0.9613 Val Loss: 0.1515 Val Acc: 0.9712
Epoch [4/15] Train Loss: 0.1286 Train Acc: 0.9564 Val Loss: 0.0927 Val Acc: 0.9519
Epoch [5/15] Train Loss: 0.0472 Train Acc: 0.9879 Val Loss: 0.0940 Val Acc: 0.9808
Epoch [6/15] Train Loss: 0.2530 Train Acc: 0.9516 Val Loss: 0.0401 Val Acc: 0.9904
Epoch [7/15] Train Loss: 0.1193 Train Acc: 0.9709 Val Loss: 0.0814 Val Acc: 0.9712
Epoch [8/15] Train Loss: 0.0448 Train Acc: 0.9879 Val Loss: 0.0116 Val Acc: 1.0000
Epoch [9/15] Train Loss: 0.0132 Train Acc: 0.9976 Val Loss: 0.0076 Val Acc: 1.0000
Epoch [10/15] Train Loss: 0.0118 Train Acc: 0.9976 Val Loss: 0.0373 Val Acc: 0.9712
Epoch [11/15] Train Loss: 0.0062 Train Acc: 1.0000 Val Loss: 0.0428 Val Acc: 0.9808
Epoch [12/15] Train Loss: 0.0035 Train Acc: 1.0000 Val Loss: 0.6980 Val Acc: 0.8846
E